In [97]:
import spacy
from numpy import dtype
from torch.distributions.constraints import positive

nlp = spacy.load("en_core_web_sm") ##You get POS tags, dependency parse, noun_chunks → ABSA ready instead of doing them manually.

In [98]:
sentences = [
    "The food was delicious but the service was slow.",
    "I loved the cozy atmosphere and friendly staff.",
    "Prices were a bit high compared to other restaurants.",
    "The pasta lacked flavor, though the portion was generous.",
    "Overall, I would recommend this place for family dinners."
]

In [99]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

def sentiment_scores(data):
    sid_obj = SentimentIntensityAnalyzer()
    compound_scores = []
    for text in data:
        sentiment_dict = sid_obj.polarity_scores(text)
        compound_scores.append(sentiment_dict['compound'])
        
        print(f"Sentence: {text}")
        print(f"Sentiment Scores: {sentiment_dict}")
        print(f"Negative Sentiment: {sentiment_dict['neg']*100:.2f}%")
        print(f"Neutral Sentiment: {sentiment_dict['neu']*100:.2f}%")
        print(f"Positive Sentiment: {sentiment_dict['pos']*100:.2f}%\n")

    # Overall sentiment: average of compound scores
    avg_compound = sum(compound_scores)/len(compound_scores)
    if avg_compound >= 0.05:
        overall = "Positive"
    elif avg_compound <= -0.05:
        overall = "Negative"
    else:
        overall = "Neutral"

    print(f"Overall Sentiment: {overall}")

In [100]:
sentiment_scores(sentences)

Sentence: The food was delicious but the service was slow.
Sentiment Scores: {'neg': 0.0, 'neu': 0.773, 'pos': 0.227, 'compound': 0.3291}
Negative Sentiment: 0.00%
Neutral Sentiment: 77.30%
Positive Sentiment: 22.70%

Sentence: I loved the cozy atmosphere and friendly staff.
Sentiment Scores: {'neg': 0.0, 'neu': 0.458, 'pos': 0.542, 'compound': 0.7964}
Negative Sentiment: 0.00%
Neutral Sentiment: 45.80%
Positive Sentiment: 54.20%

Sentence: Prices were a bit high compared to other restaurants.
Sentiment Scores: {'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}
Negative Sentiment: 0.00%
Neutral Sentiment: 100.00%
Positive Sentiment: 0.00%

Sentence: The pasta lacked flavor, though the portion was generous.
Sentiment Scores: {'neg': 0.0, 'neu': 0.708, 'pos': 0.292, 'compound': 0.5106}
Negative Sentiment: 0.00%
Neutral Sentiment: 70.80%
Positive Sentiment: 29.20%

Sentence: Overall, I would recommend this place for family dinners.
Sentiment Scores: {'neg': 0.0, 'neu': 0.762, 'pos': 0.

In [104]:
sid= SentimentIntensityAnalyzer()


def extract_aspects(doc):
    aspects = []
    for chunk in doc.noun_chunks:
        aspects.append(chunk.root.text.lower())
    return aspects


def find_opinions(doc, aspects):
    opinions = []
    for token in doc:
        # adjective modifying noun (e.g., "delicious food")
        if token.dep_ == "amod" and token.head.text.lower() in aspects:
            opinions.append((token.head.text.lower(), token.text.lower()))

        # adjective complement (e.g., "food was delicious")
        elif token.dep_ == "acomp":
            # find nominal subject of the same verb
            for child in token.head.children:
                if child.dep_ == "nsubj" and child.text.lower() in aspects:
                    opinions.append((child.text.lower(), token.text.lower()))

        # adverb modifying an adjective or verb (e.g., "bit high")
        elif token.dep_ == "advmod" and token.head.pos_ in ["ADJ", "VERB"]:
            for child in token.head.children:
                if child.dep_ == "nsubj" and child.text.lower() in aspects:
                    opinions.append((child.text.lower(), token.text.lower()))

    return opinions


def score_opinions(opinions):
    aspect_sentiments = {}
    for aspect, opinion in opinions:
        base_score = sid.polarity_scores(opinion)['compound']

        # --- Check for negation ---
        neg_factor = -1 if any(child.dep_ == "neg" for child in token.children) else 1
        # Also check if the head has negation (e.g., "was not delicious")
        if token.head:
            neg_factor *= -1 if any(child.dep_ == "neg" for child in token.head.children) else 1

        # --- Check for intensifiers ---
        intensity_factor = 1.0
        for child in token.children:
            if child.dep_ == "advmod":  # adverbs modifying the opinion word
                if child.text.lower() in ["very", "extremely", "highly", "so", "really"]:
                    intensity_factor += 0.5  # increase sentiment strength
        if token.head:
            for child in token.head.children:
                if child.dep_ == "advmod" and child.text.lower() in ["very", "extremely", "highly", "so", "really"]:
                    intensity_factor += 0.5

        # Adjusted score
        score = base_score * neg_factor * intensity_factor
        if aspect not in aspect_sentiments:
            aspect_sentiments[aspect] = []
        aspect_sentiments[aspect].append(score)
    # Aggregate sentiment per aspect
    for aspect, scores in aspect_sentiments.items():
        avg = sum(scores) / len(scores)
        sentiment = "Positive" if avg >= 0.05 else "Negative" if avg <= -0.05 else "Neutral"
        aspect_sentiments[aspect] = (sentiment, avg)
    return aspect_sentiments

def analyze_aspect_sentiment(sentences):
    results = {}
    for s in sentences:
        doc = nlp(s)
        aspects = extract_aspects(doc)
        opinions = find_opinions(doc, aspects)
        aspect_sentiments = score_opinions(opinions)
        results[s] = aspect_sentiments
    return results

analysis = analyze_aspect_sentiment(sentences)
for sentence, aspects in analysis.items():
    print(f"\nSentence: {sentence}")
    for aspect, (sentiment, score) in aspects.items():
        print(f"  Aspect: {aspect:15} | Sentiment: {sentiment:8} | Score: {score:.2f}")



Sentence: The food was delicious but the service was slow.
  Aspect: food            | Sentiment: Positive | Score: 0.57
  Aspect: service         | Sentiment: Neutral  | Score: 0.00

Sentence: I loved the cozy atmosphere and friendly staff.
  Aspect: atmosphere      | Sentiment: Neutral  | Score: 0.00
  Aspect: staff           | Sentiment: Positive | Score: 0.49

Sentence: Prices were a bit high compared to other restaurants.
  Aspect: prices          | Sentiment: Neutral  | Score: 0.00
  Aspect: restaurants     | Sentiment: Neutral  | Score: 0.00

Sentence: The pasta lacked flavor, though the portion was generous.
  Aspect: portion         | Sentiment: Positive | Score: 0.51

Sentence: Overall, I would recommend this place for family dinners.
  Aspect: i               | Sentiment: Neutral  | Score: 0.00


In [102]:
for s in sentences:
    doc_data = nlp(s)
    aspect = extract_aspects(doc_data)
    opinion = find_opinions(doc_data, aspect)
    print(f"Aspects: {aspect}")
    print(f"Opinions: {opinion}")

Aspects: ['food', 'service']
Opinions: [('food', 'delicious'), ('service', 'slow')]
Aspects: ['i', 'atmosphere', 'staff']
Opinions: [('atmosphere', 'cozy'), ('staff', 'friendly')]
Aspects: ['prices', 'restaurants']
Opinions: [('prices', 'high'), ('restaurants', 'other')]
Aspects: ['pasta', 'flavor', 'portion']
Opinions: [('portion', 'generous')]
Aspects: ['i', 'place', 'dinners']
Opinions: [('i', 'overall')]


In [103]:
doc = nlp("The food was delicious.")
for token in doc:
    print(token.text, "→", token.dep_, "→", token.head.text)

The → det → food
food → nsubj → was
was → ROOT → was
delicious → acomp → was
. → punct → was
